In [ ]:
import pandas as pd
import numpy as np

# Load Version 2 dataset
df = pd.read_csv("learntwin_student_interactions.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (10000, 10)

Columns:
['student_id', 'attempt', 'question_id', 'concept', 'difficulty', 'correct', 'time_gap_days', 'days_since_practice', 'timestamp', 'knowledge_state']


,student_id,attempt,question_id,concept,difficulty,correct,time_gap_days,days_since_practice,timestamp,knowledge_state
0,S001,1,Q043,RIGHT JOIN,0.724,0,3.007,3.007,3.007,0.3574
1,S001,2,Q452,RIGHT JOIN,0.577,0,1.471,1.471,4.478,0.3103
2,S001,3,Q022,UNION,0.499,1,1.675,6.154,6.154,0.3849
3,S001,4,Q007,WHERE,0.380,1,0.563,6.717,6.717,0.4015
4,S001,5,Q114,STORED PROCEDURES,0.598,1,2.683,9.400,9.400,0.1345


3.2 — Sort the interactions

In [ ]:
# Convert timestamp into datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Sort by student and time
df = df.sort_values(
    by=["student_id", "timestamp"]
).reset_index(drop=True)

print(df.head())

3.3 — Create previous student accuracy

In [2]:
# Total correct answers before the current attempt
df["student_prior_correct"] = (
    df.groupby("student_id")["correct"].cumsum()
    - df["correct"]
)

# Number of previous attempts
df["student_prior_attempts"] = (
    df.groupby("student_id").cumcount()
)

# Previous accuracy
df["student_prior_accuracy"] = np.where(
    df["student_prior_attempts"] > 0,
    df["student_prior_correct"] / df["student_prior_attempts"],
    0.5
)

3.4 — Create previous concept accuracy

In [3]:
# Group by student and concept
concept_group = df.groupby(
    ["student_id", "concept"]
)

# Correct answers for this concept before current attempt
df["concept_prior_correct"] = (
    concept_group["correct"].cumsum()
    - df["correct"]
)

# Previous attempts for this student and concept
df["concept_prior_attempts"] = (
    concept_group.cumcount()
)

# Previous accuracy for this concept
df["concept_prior_accuracy"] = np.where(
    df["concept_prior_attempts"] > 0,
    df["concept_prior_correct"] / df["concept_prior_attempts"],
    0.5
)

3.5 — Check the generated features

In [5]:
df[[
    "student_id",
    "concept",
    "correct",
    "student_prior_attempts",
    "student_prior_accuracy",
    "concept_prior_attempts",
    "concept_prior_accuracy"
]].head(15)

,student_id,concept,correct,student_prior_attempts,student_prior_accuracy,concept_prior_attempts,concept_prior_accuracy
0,S001,RIGHT JOIN,0,0,0.500000,0,0.5
1,S001,RIGHT JOIN,0,1,0.000000,1,0.0
2,S001,UNION,1,2,0.000000,0,0.5
3,S001,WHERE,1,3,0.333333,0,0.5
4,S001,STORED PROCEDURES,1,4,0.500000,0,0.5
5,S001,GROUP BY,0,5,0.600000,0,0.5
6,S001,TRIGGERS,0,6,0.500000,0,0.5
7,S001,STORED PROCEDURES,0,7,0.428571,1,1.0
8,S001,GROUP BY,1,8,0.375000,1,0.0
9,S001,SUBQUERIES,0,9,0.444444,0,0.5


3.6 — Select model features

In [6]:
feature_columns = [
    "concept",
    "difficulty",
    "time_gap_days",
    "days_since_practice",
    "attempt",
    "student_prior_attempts",
    "student_prior_accuracy",
    "concept_prior_attempts",
    "concept_prior_accuracy"
]

target_column = "correct"

X = df[feature_columns]
y = df[target_column]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (10000, 9)
Target shape: (10000,)


3.7 — Create a chronological train-test split

In [7]:
# First 80% attempts of every student for training
train_mask = df["attempt"] <= 80

X_train = X[train_mask]
X_test = X[~train_mask]

y_train = y[train_mask]
y_test = y[~train_mask]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8000
Testing samples: 2000


3.8 — Build the Logistic Regression model

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [9]:
categorical_features = [
    "concept"
]

numerical_features = [
    "difficulty",
    "time_gap_days",
    "days_since_practice",
    "attempt",
    "student_prior_attempts",
    "student_prior_accuracy",
    "concept_prior_attempts",
    "concept_prior_accuracy"
]

Create preprocessing:

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

Create the complete model:

In [16]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

3.9 — Train the model

In [17]:
model.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


3.10 — Make predictions

In [20]:
# Predict class: 0 or 1
y_pred = model.predict(X_test)

# Predict probability of correct answer
y_probability = model.predict_proba(X_test)[:, 1]

print("Predicted classes:")
print(y_pred[:20])

print("\nPredicted probabilities:")
print(y_probability[:20])

Predicted classes:
[0 1 0 0 1 0 0 1 1 0 0 0 0 0 1 0 0 1 0 0]

Predicted probabilities:
[0.48413475 0.53446158 0.43134323 0.40617215 0.51000281 0.37635836
 0.49386867 0.52716061 0.53038166 0.44944263 0.43795047 0.38173418
 0.34226988 0.37682845 0.52057252 0.46942468 0.3768539  0.50501481
 0.39558026 0.33033137]


3.11 — Evaluate the model

In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [19]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)
recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)
f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)
roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

Accuracy : 0.6685
Precision: 0.3766
Recall   : 0.1976
F1 Score : 0.2592
ROC-AUC  : 0.5718


3.13 — Save the trained model

In [21]:
import joblib

joblib.dump(
    model,
    "learntwin_logistic_baseline.pkl"
)

print("Model saved successfully!")

Model saved successfully!


Student interaction data
        ↓
Previous student accuracy
        ↓
Previous concept accuracy
        ↓
Time gap and difficulty
        ↓
Logistic Regression
        ↓
Predicted probability of success